# 🧊 Frostbyte — Multimodal AI Triage Pipeline

**Project:** Multimodal AI Triage Assistant for Emergency Departments  
**Hackathon:** Frostbyte (Deadline: March 15, 2026)  

This notebook implements the full multimodal pipeline:

| Stage | Model | Input → Output |
|-------|-------|-----------------|
| **Text** | Bio_ClinicalBERT | `chief_complaint` → 10 PCA features |
| **Vision** | ResNet-50 | `image_path` → 5 PCA features |
| **Fusion** | LightGBM | 22 features → ESI 1-5 prediction |
| **Explainability** | SHAP | Model → 3 interpretability plots |

**Runtime:** T4 GPU required. ~5 minutes end-to-end.

---

## 1️⃣ Environment Setup

In [ ]:
!pip install -q transformers shap lightgbm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# ============================================================
# 📁 CONFIGURE: path to your project folder on Google Drive
# ============================================================
BASE_DIR = "/content/drive/MyDrive/frostbyte"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("✅ Environment ready.")

## 2️⃣ Load Dataset & Sanity Checks

In [ ]:
CSV_PATH = os.path.join(BASE_DIR, "triage_dataset_final.csv")
df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} rows × {len(df.columns)} columns")
print(f"\nESI Distribution:\n{df['target_esi'].value_counts().sort_index()}")
print(f"\nRows with images: {(df['image_path'] != 'None').sum()}")
print(f"Unique complaints: {df['chief_complaint'].nunique()}")

df[["patient_id", "chief_complaint", "target_esi", "image_path"]].head(10)

## 3️⃣ Text Modality — ClinicalBERT Embeddings + PCA

**Pipeline:** Tokenize `chief_complaint` → Bio_ClinicalBERT forward pass → extract `[CLS]` 768-d → PCA to 10 components

**Why PCA to 10?** Prevents 768 text dimensions from overpowering the 7 tabular vitals during fusion.

In [ ]:
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

print(f"Loading {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model = bert_model.to(device).eval()
print(f"Model loaded on {device}")

In [ ]:
complaints = df["chief_complaint"].fillna("Unknown").tolist()

BATCH_SIZE = 16
all_embeddings = []

print(f"Extracting embeddings for {len(complaints)} complaints …")

with torch.no_grad():
    for i in range(0, len(complaints), BATCH_SIZE):
        batch_texts = complaints[i : i + BATCH_SIZE]

        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt",
        ).to(device)

        outputs = bert_model(**tokens)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

        if (i // BATCH_SIZE) % 20 == 0:
            print(f"  Processed {min(i + BATCH_SIZE, len(complaints))}/{len(complaints)}")

embeddings_matrix = np.vstack(all_embeddings)
print(f"\nRaw embedding matrix: {embeddings_matrix.shape}")

# PCA: 768 → 10
N_TEXT_COMPONENTS = 10
pca_text = PCA(n_components=N_TEXT_COMPONENTS, random_state=42)
text_pca = pca_text.fit_transform(embeddings_matrix)

text_cols = [f"text_feat_{i}" for i in range(N_TEXT_COMPONENTS)]
for idx, col in enumerate(text_cols):
    df[col] = text_pca[:, idx]

print(f"PCA explained variance: {np.round(pca_text.explained_variance_ratio_, 4)}")
print(f"Total: {pca_text.explained_variance_ratio_.sum():.4f}")
print(f"\n✅ Added {N_TEXT_COMPONENTS} text features. Shape: {df.shape}")

# Checkpoint save
df.to_csv(os.path.join(BASE_DIR, "triage_with_text_features.csv"), index=False)
print("💾 Checkpoint: triage_with_text_features.csv")

## 4️⃣ Vision Modality — ResNet-50 Embeddings + PCA

**Pipeline:** Load pre-trained ResNet-50 → strip final FC layer → extract 2048-d embeddings → PCA to 5 components

**Key design:** PCA is fitted on ONLY the 28 rows with real images. Zero-padded rows stay exactly zero to avoid mean-centering artifacts.

In [ ]:
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm

print("Initializing ResNet-50 (Transfer Learning Mode)...")
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)
resnet.fc = torch.nn.Identity()
resnet.eval()
resnet.to(device)

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Extracting visual embeddings on {device}...")
image_embeddings = []
has_image_mask = []
images_found = 0

for path in tqdm(df['image_path'], desc="Processing images"):
    if path == "None" or pd.isna(path):
        image_embeddings.append(np.zeros(2048))
        has_image_mask.append(False)
    else:
        try:
            full_path = os.path.join(BASE_DIR, path)
            img = Image.open(full_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                emb = resnet(img_tensor).cpu().numpy().flatten()
            image_embeddings.append(emb)
            has_image_mask.append(True)
            images_found += 1
        except Exception as e:
            print(f"\n⚠️ Could not load {path}: {e}")
            image_embeddings.append(np.zeros(2048))
            has_image_mask.append(False)

image_embeddings = np.array(image_embeddings)
has_image_mask = np.array(has_image_mask)
print(f"\nExtracted features for {images_found} real images.")

# PCA: 2048 → 5 (fit on real images ONLY)
N_IMG_COMPONENTS = 5
pca_img = PCA(n_components=N_IMG_COMPONENTS, random_state=42)
pca_img.fit(image_embeddings[has_image_mask])

img_pca = np.zeros((len(df), N_IMG_COMPONENTS))
img_pca[has_image_mask] = pca_img.transform(image_embeddings[has_image_mask])

img_cols = [f"img_feat_{i}" for i in range(N_IMG_COMPONENTS)]
for i, col in enumerate(img_cols):
    df[col] = img_pca[:, i]

print(f"PCA variance (real images): {np.round(pca_img.explained_variance_ratio_, 4)}")
print(f"Total: {pca_img.explained_variance_ratio_.sum():.4f}")
print(f"\n✅ Added {N_IMG_COMPONENTS} image features. Shape: {df.shape}")

# Save master multimodal CSV
df.to_csv(os.path.join(BASE_DIR, "triage_master_multimodal.csv"), index=False)
print("💾 Saved: triage_master_multimodal.csv")

## 5️⃣ Late-Fusion Meta-Model (LightGBM)

**The Brain:** A single LightGBM classifier on the concatenated 22-feature space.

Uses `class_weight='balanced'` to respect the clinical ESI distribution from MIMIC-IV-ED.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

tabular_features = ["age", "heart_rate", "resp_rate", "spo2", "temp_f", "systolic_bp", "pain_scale"]
ALL_FEATURES = tabular_features + text_cols + img_cols
print(f"Feature space: {len(tabular_features)} tabular + {len(text_cols)} text + {len(img_cols)} image = {len(ALL_FEATURES)} total")

X = df[ALL_FEATURES]
y = df["target_esi"] - 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

print("\nTraining Late-Fusion Meta-Model...")
model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

ESI_NAMES = ["ESI 1 (Resuscitation)", "ESI 2 (Emergent)", "ESI 3 (Urgent)", "ESI 4 (Less Urgent)", "ESI 5 (Non-Urgent)"]

print(f"\n{'='*50}")
print(f"  MULTIMODAL FUSION ACCURACY: {acc:.4f}")
print(f"{'='*50}\n")
print(classification_report(y_test, y_pred, target_names=ESI_NAMES))

## 6️⃣ SHAP Explainability Engine

Three visualizations for the hackathon pitch:
1. **Global bar chart** — which features matter most across all classes
2. **ESI-1 beeswarm** — directional feature impact for critical patients
3. **Local waterfall** — single-patient "why ESI 1?" explanation

In [ ]:
import shap
import matplotlib.pyplot as plt

print("Generating SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Detect SHAP output format (varies by library version)
if isinstance(shap_values, list):
    sv_esi1 = shap_values[0]
    sv_esi1_patient = lambda idx: shap_values[0][idx]
    base_val_esi1 = explainer.expected_value[0]
else:
    sv_esi1 = shap_values[:, :, 0]
    sv_esi1_patient = lambda idx: shap_values[idx, :, 0]
    base_val_esi1 = explainer.expected_value[0]

print(f"SHAP output shape: {np.array(shap_values).shape}")

In [ ]:
# Plot 1: Global Feature Importance
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False, feature_names=ALL_FEATURES)
plt.title("Global Feature Importance — Multimodal Triage Model", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "shap_global_importance.png"), dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: shap_global_importance.png")

In [ ]:
# Plot 2: ESI-1 Beeswarm
plt.figure(figsize=(12, 8))
shap.summary_plot(sv_esi1, X_test, show=False, feature_names=ALL_FEATURES)
plt.title("SHAP Beeswarm — ESI 1 (Resuscitation) Class", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "shap_esi1_beeswarm.png"), dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: shap_esi1_beeswarm.png")

In [ ]:
# Plot 3: Local Waterfall for a Critical Patient
esi1_indices = np.where(y_pred == 0)[0]
if len(esi1_indices) > 0:
    critical_idx = esi1_indices[0]
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=sv_esi1_patient(critical_idx),
            base_values=base_val_esi1,
            data=X_test.iloc[critical_idx],
            feature_names=ALL_FEATURES,
        ),
        show=False,
    )
    plt.title(f"Patient #{critical_idx} — Why ESI 1 (Resuscitation)?", fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "shap_critical_patient.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Saved: shap_critical_patient.png")
else:
    print("⚠️ No ESI 1 predictions in test set.")

## 7️⃣ Save Model Artifacts

In [ ]:
model_path = os.path.join(BASE_DIR, "triage_multimodal_model.pkl")
joblib.dump(model, model_path)
print(f"💾 Model (pickle): {model_path}")

lgb_path = os.path.join(BASE_DIR, "triage_multimodal_model.txt")
model.booster_.save_model(lgb_path)
print(f"💾 Model (LightGBM native): {lgb_path}")

print(f"\n{'='*50}")
print("  🎉 FULL MULTIMODAL PIPELINE COMPLETE!")
print(f"{'='*50}")
print(f"\nDataset:  {df.shape[0]} patients × {df.shape[1]} columns")
print(f"Features: 7 tabular + 10 text (ClinicalBERT) + 5 image (ResNet-50) = 22")
print(f"Accuracy: {acc:.4f}")
print(f"\nArtifacts saved to Drive:")
print(f"  • triage_master_multimodal.csv")
print(f"  • triage_multimodal_model.pkl / .txt")
print(f"  • shap_global_importance.png")
print(f"  • shap_esi1_beeswarm.png")
print(f"  • shap_critical_patient.png")